# **Boosting Wav2Vec2 with n-grams in 🤗 Transformers**

**Wav2Vec2** is a popular pre-trained model for speech recognition. Released in [September 2020](https://ai.facebook.com/blog/wav2vec-20-learning-the-structure-of-speech-from-raw-audio/) by Meta AI Research, the novel architecture catalyzed progress in self-supervised pretraining for speech recognition, *e.g.* [*G. Ng et al.*, 2021](https://arxiv.org/pdf/2104.03416.pdf), [*Chen et al*, 2021](https://arxiv.org/abs/2110.13900), [*Hsu et al.*, 2021](https://arxiv.org/abs/2106.07447) and [*Babu et al.*, 2021](https://arxiv.org/abs/2111.09296). On the Hugging Face Hub, Wav2Vec2's most popular pre-trained checkpoint currently amounts to over [**250,000** monthly downloads](https://huggingface.co/facebook/wav2vec2-base-960h).

Using Connectionist Temporal Classification (CTC), pre-trained Wav2Vec2-like checkpoints are extremely easy to fine-tune on downstream speech recognition tasks.
In a nutshell, fine-tuning pre-trained Wav2Vec2 checkpoints works as follows:

A single randomly initialized linear layer is stacked on top of the pre-trained checkpoint and trained to classify raw audio input to a sequence of letters. It does so by:

1.  extracting audio representations from the raw audio (using CNN layers),
2. processing the sequence of audio representations with a stack of transformer layers, and,
3. classifying the processed audio representations into a sequence of output letters.

Previously audio classification models required an additional language model (LM) and a dictionary to transform the sequence of classified audio frames to a coherent transcription.
Wav2Vec2's architecture is based on transformer layers, thus giving each processed audio representation context
from all other audio representations. In addition,
Wav2Vec2 leverages the [CTC algorithm](https://distill.pub/2017/ctc/) for fine-tuning, which solves the problem of alignment between a varying "input audio length"-to-"output text length" ratio.

Having contextualized audio classifications and no alignment problems, Wav2Vec2 does not require
an external language model or dictionary to yield acceptable audio transcriptions.

As can be seen in Appendix C of the [official paper](https://arxiv.org/abs/2006.11477), Wav2Vec2 gives impressive downstream performances on [LibriSpeech]((https://huggingface.co/datasets/librispeech_asr)) without using a language model at all. However, from the appendix, it also becomes clear that using Wav2Vec2 in combination with a language model can yield a significant improvement, especially when the model was trained on only 10 minutes of transcribed audio.

Until recently, the 🤗 Transformers library did not offer a simple user interface to decode audio files with a fine-tuned Wav2Vec2 **and** a language model. This has thankfully changed. 🤗 Transformers now offers an easy-to-use integration with *Kensho Technologies'* [pyctcdecode library](https://github.com/kensho-technologies/pyctcdecode). This blog post is a step-by-step **technical** guide to explain how one can create an **n-gram** language model and combine it with an existing fine-tuned Wav2Vec2 checkpoint using 🤗 Datasets and 🤗 Transformers.

We start by:

1. How does decoding audio with an LM differ from decoding audio without an LM?
2. How to get suitable data for a language model?
3. How to build an *n-gram* with KenLM?
4. How to combine the *n-gram* with a fine-tuned Wav2Vec2 checkpoint?

For a deep dive into how Wav2Vec2 functions - which is not necessary for this blog post - the reader is advised to consult the following material:

- [wav2vec 2.0: A Framework for Self-Supervised Learning of Speech Representations](https://arxiv.org/abs/2006.11477)
- [Fine-Tune Wav2Vec2 for English ASR with 🤗 Transformers](https://huggingface.co/blog/fine-tune-wav2vec2-english)
- [An Illustrated Tour of Wav2vec 2.0](https://jonathanbgn.com/2021/09/30/illustrated-wav2vec-2.html)

## **1. Decoding audio data with Wav2Vec2 and a language model**

As shown in 🤗 Transformers [exemple docs of Wav2Vec2](https://huggingface.co/docs/transformers/master/en/model_doc/wav2vec2#transformers.Wav2Vec2ForCTC), audio can be transcribed as follows.

We install `datasets` and `transformers` as well as `pyctcdecode` and `kenLM`'s Python bindings to be able to run the language model integration.



In [ ]:
!pip install https://github.com/kpu/kenlm/archive/master.zip  --q
# !pip install datasets==2.0.0

     \ 553.6 kB 9.4 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


Let's load a small excerpt of the [Librispeech dataset](https://huggingface.co/datasets/librispeech_asr) to demonstrate Wav2Vec2's speech transcription capabilities.

In [ ]:
# from datasets import load_dataset

# dataset = load_dataset("hf-internal-testing/librispeech_asr_demo", "clean", split="validation")
# dataset

We can pick one of the 73 audio samples and listen to it.

In [ ]:
# import IPython.display as ipd

# audio_sample = dataset[2]
# print(audio_sample["text"].lower())
# ipd.Audio(data=audio_sample["audio"]["array"], autoplay=True, rate=audio_sample["audio"]["sampling_rate"])

Having chosen a data sample, we now load the fine-tuned model and processor.

In [ ]:
# from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

# processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base-100h")
# model = Wav2Vec2ForCTC.from_pretrained("facebook/wav2vec2-base-100h")

Next, we process the data

In [ ]:
# inputs = processor(audio_sample["audio"]["array"], sampling_rate=16_000, return_tensors="pt")

forward it to the model

In [ ]:
# import torch

# with torch.no_grad():
#   logits = model(**inputs).logits

and decode it

In [ ]:
# predicted_ids = torch.argmax(logits, dim=-1)
# transcription = processor.batch_decode(predicted_ids)

# transcription[0].lower()

Comparing the transcription to the target transcription above, we can see that some words *sound* correct, but are not *spelled* correctly, *e.g.*:

- *christmaus* vs. *christmas*
- *rose* vs. *roast*
- *simalyis* vs. *similes*

Let's see whether combining Wav2Vec2 with an ***n-gram*** lnguage model can help here.

For demonstration purposes, we have prepared a new model repository [patrickvonplaten/wav2vec2-base-100h-with-lm](https://huggingface.co/patrickvonplaten/wav2vec2-base-100h-with-lm) which contains the same Wav2Vec2 checkpoint but has an additional **4-gram** language model for English.

Instead of using `Wav2Vec2Processor`, this time we use `Wav2Vec2ProcessorWithLM` to load the **4-gram** model in addition to the feature extractor and tokenizer.

In [ ]:
# from transformers import Wav2Vec2ProcessorWithLM

# processor = Wav2Vec2ProcessorWithLM.from_pretrained("patrickvonplaten/wav2vec2-base-100h-with-lm")

In constrast to decoding the audio without language model, the processor now directly receives the model's output `logits` instead of the `argmax(logits)` (called `predicted_ids`) above. The reason is that when decoding with a language model, at each time step, the processor takes the probabilities of all possible output characters into account. Let's take a look at the dimension of the `logits` output.

In [ ]:
# logits.shape

We can see that the `logits` correspond to a sequence of 624 vectors each having 32 entries. Each of the 32 entries thereby stands for the logit probability of one of the 32 possible output characters of the model:

In [ ]:
# " ".join(sorted(processor.tokenizer.get_vocab()))

Intuitively, one can understand the decoding process of `Wav2Vec2ProcessorWithLM` as applying beam search through a matrix of size 624 $\times$ 32 probabilities while leveraging the probabilities of the next letters as given by the *n-gram* language model.

OK, let's run the decoding step again. `pyctcdecode` language model decoder does not automatically convert `torch` tensors to `numpy` so we'll have to convert them ourselves before.

In [ ]:
# transcription = processor.batch_decode(logits.numpy()).text
# transcription[0].lower()

Cool! Recalling the words `facebook/wav2vec2-base-100h` without a language model transcribed incorrectly previously, *e.g.*,

> - *christmaus* vs. *christmas*
- *rose* vs. *roast*
- *simalyis* vs. *similes*

we can take another look at the transcription of `facebook/wav2vec2-base-100h` **with** a 4-gram language model. 2 out of 3 errors are corrected; *christmas* and *similes* have been correctly transcribed.

Interestingly, the incorrect transcription of *rose* persists. However, this should not surprise us very much. Decoding audio without a language model is much more prone to yield spelling mistakes, such as *christmaus* or *similes* (those words don't exist in the English language as far as I know). This is because the speech recognition system almost solely bases its prediction on the acoustic input it was given and not really on the language modeling context of previous and successive predicted letters ${}^1$.
If on the other hand, we add a language model, we can be fairly sure that the speech recognition system will heavily reduce spelling errors since a well-trained *n-gram* model will surely not predict a word that has spelling errors. But the word *rose* is a valid English word and therefore the 4-gram will predict this word with a probability that is not insignificant.

The language model on its own most likely does favor the correct word *roast* since the word sequence *roast beef* is much more common in English than *rose beef*. Because the final transcription is derived from a weighted combination of `facebook/wav2vec2-base-100h` output probabilities and those of the *n-gram* language model, it is quite common to see incorrectly transcribed words such as *rose*.

For more information on how you can tweak different parameters when decoding with `Wav2Vec2ProcessorWithLM`, please take a look at the official documentation [here](https://huggingface.co/docs/transformers/master/en/model_doc/wav2vec2#transformers.Wav2Vec2ProcessorWithLM.batch_decode).

---
${}^1$ Some research shows that a model such as `facebook/wav2vec2-base-100h` - when sufficiently large and trained on enough data - can learn language modeling dependencies between intermediate audio representations similar to a language model.


Great, now that you have seen the advantages adding an *n-gram* language model can bring, let's dive into how to create an *n-gram* and `Wav2Vec2ProcessorWithLM` from scratch.

## **2. Getting data for your language model**

A language model that is useful for a speech recognition system should support the acoustic model, *e.g.* Wav2Vec2, in predicting the next word (or token, letter) and therefore model the following distribution:

$\mathbf{P}(w_n | \mathbf{w}_0^{t-1})$ with $w_n$ being the next word and $\mathbf{w}_0^{t-1}$ being the sequence of all previous words since the beginning of the utterance. Simply said, the language model should be good at predicting the next word given all previously transcribed words regardless of the audio input given to the speech recognition system.

As always a language model is only as good as the data it is trained on. In the case of speech recognition, we should therefore ask ourselves for what kind of data, the speech recognition will be used for: *conversations*, *audiobooks*, *movies*, *speeches*, *, etc*, ...?

The language model should be good at modeling language that corresponds to the
target transcriptions of the speech recognition system.
For demonstration purposes, we assume here that we have fine-tuned a pre-trained [`facebook/wav2vec2-xls-r-300m`](https://huggingface.co/facebook/wav2vec2-xls-r-300m) on [Common Voice 7](https://huggingface.co/datasets/mozilla-foundation/common_voice_7_0) in Swedish. The fine-tuned checkpoint can
be found [here](https://huggingface.co/hf-test/xls-r-300m-sv).
Common Voice 7 is a relatively crowd-sourced read-out audio dataset and we will evaluate the model on its test data.

Let's now look for suitable text data on the Hugging Face Hub. We search all datasets for those [that contain Swedish data](https://huggingface.co/datasets?languages=languages:sv&sort=downloads).
Browsing a bit through the datasets, we are looking for a dataset that is similar to Common Voice's read-out audio data. The obvious choices of [oscar](https://huggingface.co/datasets/oscar) and [mc4](https://huggingface.co/datasets/mc4) might not be the most suitable here because they:

- are generated from crawling the web, which might not be very clean and correspond well to spoken language
- require a lot of pre-processing
- are very large which is not ideal for demonstration purposes here 😉

A dataset that seems sensible here and which is relatively clean and easy to pre-process is [europarl_bilingual](https://huggingface.co/datasets/europarl_bilingual) as it's a dataset that is based on discussions and talks of the European parliament. It should therefore be relatively clean and correspond well to read-out audio data. The dataset is originally designed for machine translation and can therefore only be accessed in translation pairs. We will only extract the text of the target language, Swedish (`sv`), from the *English-to-Swedish* translations.

Let's download the data.

We see that the data is quite large - it has over a million translations. Since it's only text data, it should be relatively easy to process though.

Next, let's look at how the data was preprocessed when training the fine-tuned *XLS-R* checkpoint in Swedish. Looking at the [`run.sh` file](https://huggingface.co/hf-test/xls-r-300m-sv/blob/main/run.sh), we can see that the following characters were removed from the official transcriptions:

Let's do the same here so that the alphabet of our language model matches the one of the fine-tuned acoustic checkpoints.

We can write a single map function to extract the Swedish text and process it right away.

Let's apply the `.map()` function. This should take roughly 5 minutes.

Great. Our dataset is already finished. Let's upload it to the Hub so that we can inspect and reuse it better.

You can log in by executing the following cell.

In [ ]:
# !pip install huggingface_hub --upgrade
# !pip install datasets --upgrade


In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

login(HF_TOKEN)

Next, we call 🤗 Hugging Face's [`push_to_hub`](https://huggingface.co/docs/datasets/package_reference/main_classes.html?highlight=push#datasets.Dataset.push_to_hub) method to upload the dataset to the repo `"swedish_corpora_parliament_processed"`.

That was easy! The dataset viewer is automatically enabled when uploading a new dataset, which is very convenient. You can now directly inspect the dataset online.

Feel free to look through our preprocessed dataset directly on [`hf-test/sv_corpora_parliament_processed`](https://huggingface.co/datasets/hf-test/sv_corpora_parliament_processed). Even if we are not a native speaker in Swedish, we can see that the data is well processed and seems clean.

Next, let's use the data to build a language model.

## **3. Build an *n-gram* with KenLM**

While large language models based on the [Transformer architecture](https://jalammar.github.io/illustrated-transformer/) have become the standard in NLP, it is still very common to use an ***n-gram*** LM to boost speech recognition systems - as shown in Section 1.

Looking again at Table 9 of Appendix C of the [official Wav2Vec2 paper](https://arxiv.org/abs/2006.11477), it can be noticed that using a *Transformer*-based LM for decoding clearly yields better results than using an *n-gram* model, but the difference between *n-gram* and *Transformer*-based LM is much less significant than the difference between *n-gram* and no LM.

*E.g.*, for the large Wav2Vec2 checkpoint that was fine-tuned on 10min only, an *n-gram* reduces the word error rate (WER) compared to no LM by *ca.* 80% while a *Transformer*-based LM *only* reduces the WER by another 23% compared to the *n-gram*. This relative WER reduction becomes less, the more data the acoustic model has been trained on. *E.g.*, for the large checkpoint a *Transformer*-based LM reduces the WER by merely 8% compared to an *n-gram* LM whereas the *n-gram* still yields a 21% WER reduction compared to no language model.

The reason why an *n-gram* is preferred over a *Transformer*-based LM is that *n-grams* come at a significantly smaller computational cost. For an *n-gram*, retrieving the probability of a word given previous words is almost only as computationally expensive as querying a look-up table or tree-like data storage - *i.e.* it's very fast compared to modern *Transformer*-based language models that would require a full forward pass to retrieve the next word probabilities.

For more information on how *n-grams* function and why they are (still) so useful for speech recognition, the reader is advised to take a look at [this excellent summary](https://web.stanford.edu/~jurafsky/slp3/3.pdf) from Stanford.

Great, let's see step-by-step how to build an *n-gram*. We will use the popular [KenLM library](https://github.com/kpu/kenlm) to do so. Let's start by installing the Ubuntu library prerequisites:

In [ ]:
!sudo apt install build-essential cmake libboost-system-dev libboost-thread-dev libboost-program-options-dev libboost-test-dev libeigen3-dev zlib1g-dev libbz2-dev liblzma-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.9ubuntu3).
libbz2-dev is already the newest version (1.0.8-5build1).
libbz2-dev set to manually installed.
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
liblzma-dev is already the newest version (5.2.5-2ubuntu1.1).
liblzma-dev set to manually installed.
zlib1g-dev is already the newest version (1:1.2.11.dfsg-2ubuntu9.2).
zlib1g-dev set to manually installed.
The following additional packages will be installed:
  libboost-atomic1.74-dev libboost-atomic1.74.0 libboost-chrono1.74-dev
  libboost-chrono1.74.0 libboost-date-time1.74-dev libboost-date-time1.74.0
  libboost-program-options1.74-dev libboost-program-options1.74.0
  libboost-serialization1.74-dev libboost-serialization1.74.0
  libboost-system1.74-dev libboost-system1.74.0 libboost-test1.74-dev
  libboost-test1.74.0 libboost-thread1.74-dev libboost-thread1.74.0
Suggeste

before downloading and unpacking the KenLM repo.

In [ ]:
!wget -O - https://kheafield.com/code/kenlm.tar.gz | tar xz

--2026-08-05 15:48:34--  https://kheafield.com/code/kenlm.tar.gz
Resolving kheafield.com (kheafield.com)... 129.80.89.152, 2603:c020:4009:8710:ca:11:17:0
Connecting to kheafield.com (kheafield.com)|129.80.89.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 491888 (480K) [application/octet-stream]
Saving to: ‘STDOUT’

-                   100%[===================>] 480.36K  --.-KB/s    in 0.05s   

2026-08-05 15:48:34 (8.82 MB/s) - written to stdout [491888/491888]



KenLM is written in C++, so we'll make use of `cmake` to build the binaries.

In [ ]:
!mkdir kenlm/build && cd kenlm/build && cmake .. && make -j2
!ls kenlm/build/bin

CMake Deprecation Warning at CMakeLists.txt:1 (cmake_minimum_required):
  Compatibility with CMake < 3.10 will be removed from a future version of
  CMake.

  Update the VERSION argument <min> value.  Or, use the <min>...<max> syntax
  to tell CMake that the project requires at least <min> but has been updated
  to work with policies introduced by <max> or earlier.


-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMake Warning (dev) at CMakeLists.txt:85 (find_package):
  Policy CMP0167 is not set: The FindBoost module is removed.  Run "

Great, as we can see, the executable functions have successfully been built under `kenlm/build/bin/`.

KenLM by default computes an *n-gram* with [Kneser-Ney smooting](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing). All text data used to create the *n-gram* is expected to be stored in a text file.
We download our dataset and save it as a `.txt` file.

In [ ]:
from datasets import load_dataset
import datasets

username = "MahmoodAnaam" # change to your username

train_dataset = load_dataset(f"{username}/LRS2-Text",split="train")
pretrain_dataset = load_dataset(f"{username}/LRS2-Text",split="pretrain")

dataset = datasets.concatenate_datasets([train_dataset, pretrain_dataset])
dataset

README.md:   0%|          | 0.00/639 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.54MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/pretrain-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.27MB            

data/pretrain-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/45814 [00:00<?, ? examples/s]

Generating pretrain split:   0%|          | 0/89439 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1082 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1243 [00:00<?, ? examples/s]

Dataset({
    features: ['sample_id', 'text'],
    num_rows: 135253
})

In [ ]:
with open("text.txt", "w") as file:
  file.write(" ".join(dataset["text"]))

Now, we just have to run KenLM's `lmplz` command to build our *n-gram*, called `"5gram.arpa"`. As it's relatively common in speech recognition, we build a *5-gram* by passing the `-o 5` parameter.
For more information on the different *n-gram* LM that can be built
with KenLM, one can take a look at the [official website of KenLM](https://kheafield.com/code/kenlm/).

Executing the command below might take a minute or so.

In [ ]:

!kenlm/build/bin/lmplz -o 3 <"text.txt" > "3gram.arpa"

=== 1/5 Counting and sorting n-grams ===
Reading /content/text.txt
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Unigram tokens 1975130 types 37490
=== 2/5 Calculating and sorting adjusted counts ===
Chain sizes: 1:449880 2:3785811456 3:7098396672
Statistics:
1 37489 D1=0.586598 D2=1.02742 D3+=1.53218
2 500060 D1=0.763133 D2=1.11746 D3+=1.39119
3 1224911 D1=0.756811 D2=1.48251 D3+=1.53847
Memory estimate for binary LM:
type       kB
probing 34203 assuming -p 1.5
probing 37280 assuming -r models -p 1.5
trie    14010 without quantization
trie     7705 assuming -q 8 -b 8 quantization 
trie    13230 assuming -a 22 array pointer compression
trie     6925 assuming -a 22 -q 8 -b 8 array pointer compression and quantization
=== 3/5 Calculating and sorting initial probabilities ===
Chain sizes: 1:449868 2:8000960 3:24498220
----5---10---15--

Great, we have built a *5-gram* LM! Let's inspect the first couple of lines.

In [ ]:
!head -20 3gram.arpa

\data\
ngram 1=37489
ngram 2=500060
ngram 3=1224911

\1-grams:
-5.676108	<unk>	0
0	<s>	-0.117399506
-2.4896352	WHEN	-0.9442886
-2.798916	YOU'RE	-0.5910898
-3.7931075	COOKING	-0.2631588
-4.2921076	CHIPS	-0.22100213
-2.3278577	AT	-0.7096658
-3.2269652	HOME	-0.47997788
-1.915796	THE	-0.6264068
-3.9243445	TRADITIONAL	-0.15095766
-4.3464155	CHIP	-0.20386726
-4.676924	PAN	-0.15235561
-3.603776	OFTEN	-0.27794796
-4.2921076	STAYS	-0.28940672


There is a small problem that 🤗 Transformers will not be happy about later on.
The *5-gram* correctly includes a "Unknown" or `<unk>`, as well as a *begin-of-sentence*, `<s>` token, but no *end-of-sentence*, `</s>` token.
This sadly has to be corrected currently after the build.

We can simply add the *end-of-sentence* token by adding the line `0 </s>  -0.11831701` below the *begin-of-sentence* token and increasing the `ngram 1` count by 1. Because the file has roughly 100 million lines, this command will take *ca.* 2 minutes.

In [ ]:
with open("3gram.arpa", "r") as read_file, open("3gram_correct.arpa", "w") as write_file:
  has_added_eos = False
  for line in read_file:
    if not has_added_eos and "ngram 1=" in line:
      count=line.strip().split("=")[-1]
      write_file.write(line.replace(f"{count}", f"{int(count)+1}"))
    elif not has_added_eos and "<s>" in line:
      write_file.write(line)
      write_file.write(line.replace("<s>", "</s>"))
      has_added_eos = True
    else:
      write_file.write(line)

Let's now inspect the corrected *5-gram*.

In [ ]:
!head -20 3gram_correct.arpa

\data\
ngram 1=37490
ngram 2=500060
ngram 3=1224911

\1-grams:
-5.676108	<unk>	0
0	<s>	-0.117399506
0	</s>	-0.117399506
-2.4896352	WHEN	-0.9442886
-2.798916	YOU'RE	-0.5910898
-3.7931075	COOKING	-0.2631588
-4.2921076	CHIPS	-0.22100213
-2.3278577	AT	-0.7096658
-3.2269652	HOME	-0.47997788
-1.915796	THE	-0.6264068
-3.9243445	TRADITIONAL	-0.15095766
-4.3464155	CHIP	-0.20386726
-4.676924	PAN	-0.15235561
-3.603776	OFTEN	-0.27794796


Great, this looks better! We're done at this point and all that is left to do is to correctly integrate the `"ngram"` with [`pyctcdecode`](https://github.com/kensho-technologies/pyctcdecode) and 🤗 Transformers.

## **4. Combine an *n-gram* with Wav2Vec2**

In a final step, we want to wrap the *5-gram* into a `Wav2Vec2ProcessorWithLM` object to make the *5-gram* boosted decoding as seamless as shown in Section 1.
We start by downloading the currently "LM-less" processor of [`xls-r-300m-sv`](https://huggingface.co/hf-test/xls-r-300m-sv).

In [ ]:
!pip install pyctcdecode --q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 60.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires nump

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("MahmoodAnaam/MSP-Processor-With-LM",trust_remote_code=True)

processor_config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

processing_msp_with_lm.py:   0%|          | 0.00/27.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/MahmoodAnaam/MSP-Processor-With-LM:
- processing_msp_with_lm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


feature_extraction_msp_audio.py:   0%|          | 0.00/5.74k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/MahmoodAnaam/MSP-Processor-With-LM:
- feature_extraction_msp_audio.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

[transformers] A new version of the following files was downloaded from https://huggingface.co/MahmoodAnaam/MSP-Processor-With-LM:
- video_processing_msp_visual.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Next, we extract the vocabulary of its tokenizer as it represents the `"labels"` of `pyctcdecode`'s `BeamSearchDecoder` class.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
"MahmoodAnaam/MSP-ASR"
)

config.json:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

The repository MahmoodAnaam/MSP-ASR contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/MahmoodAnaam/MSP-ASR .
 You can inspect the repository content at https://hf.co/MahmoodAnaam/MSP-ASR.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


configuration_msp_audio.py:   0%|          | 0.00/6.14k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/MahmoodAnaam/MSP-ASR:
- configuration_msp_audio.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json:   0%|          | 0.00/1.44k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

In [ ]:
vocab_dict = tokenizer.get_vocab()
sorted_vocab_dict = {k: v for k, v in sorted(vocab_dict.items(), key=lambda item: item[1])}

In [ ]:
sorted_vocab_dict

{'<pad>': 0,
 '<s>': 1,
 '</s>': 2,
 '<unk>': 3,
 '|': 4,
 'E': 5,
 'T': 6,
 'O': 7,
 'A': 8,
 'I': 9,
 'N': 10,
 'H': 11,
 'S': 12,
 'R': 13,
 'L': 14,
 'D': 15,
 'U': 16,
 'Y': 17,
 'W': 18,
 'M': 19,
 'C': 20,
 'G': 21,
 'F': 22,
 'P': 23,
 'B': 24,
 'K': 25,
 "'": 26,
 'V': 27,
 'J': 28,
 'X': 29,
 'Q': 30,
 'Z': 31}

The `"labels"` and the previously built `5gram_correct.arpa` file is all that's needed to build the decoder.

In [ ]:
from pyctcdecode import build_ctcdecoder

decoder = build_ctcdecoder(
    labels=list(sorted_vocab_dict.keys()),
    kenlm_model_path="3gram_correct.arpa",
)

We can safely ignore the warning and all that is left to do now is to wrap the just created `decoder`, together with the processor's `tokenizer` and `feature_extractor` into a `Wav2Vec2ProcessorWithLM` class.

In [ ]:
from transformers import Wav2Vec2ProcessorWithLM

processor_class = processor.__class__
processor_with_lm = processor_class(
    feature_extractor=processor.feature_extractor,
    video_processor=processor.video_processor,
    tokenizer=tokenizer,
    decoder=decoder
)

We want to directly upload the LM-boosted processor into
the model folder of [`xls-r-300m-sv`](https://huggingface.co/hf-test/xls-r-300m-sv) to have all relevant files in one place.

Let's clone the repo, add the new decoder files and upload them afterward.
First, we need to install `git-lfs`.

In [ ]:
!sudo apt-get install git-lfs tree

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  git-lfs tree
0 upgraded, 2 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,592 kB of archives.
After this operation, 10.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 git-lfs amd64 3.0.2-1ubuntu0.3 [3,544 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tree amd64 2.0.2-1 [47.9 kB]
Fetched 3,592 kB in 4s (884 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open st

Cloning and uploading of modeling files can be done conveniently with the `huggingface_hub`'s `Repository` class.

More information on how to use the `huggingface_hub` to upload any files, please take a look at the [official docs](https://huggingface.co/docs/hub/how-to-upstream).

In [ ]:
!hf auth login

Hint: A new version of huggingface_hub (1.26.0) is available! You are using version 1.23.0.
To update, run: hf update
User is already logged in. Use `hf auth login --force` to force re-login.


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="MahmoodAnaam/MSP-Processor-With-LM-LRS2",
                local_dir="MSP-Processor-With-LM-LRS2"
                )

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

'/content/MSP-Processor-With-LM-LRS2'

Having cloned `xls-r-300m-sv`, let's save the new processor with LM into it.

In [ ]:
processor_with_lm.save_pretrained("MSP-Processor-With-LM-LRS2")

Let's inspect the local repository. The `tree` command conveniently can also show the size of the different files.

In [ ]:
!tree -h MSP-Processor-With-LM-LRS2/

[4.0K]  MSP-Processor-With-LM-LRS2/
├── [ 198]  alphabet.json
├── [5.6K]  feature_extraction_msp_audio.py
├── [4.0K]  language_model
│   ├── [ 50M]  3gram_correct.arpa
│   ├── [  78]  attrs.json
│   └── [308K]  unigrams.txt
├── [ 26K]  processing_msp_with_lm.py
├── [1.2K]  processor_config.json
├── [1.4K]  tokenizer_config.json
├── [5.3K]  video_processing_msp_visual.py
└── [ 358]  vocab.json

1 directory, 10 files


As can be seen the *5-gram* LM is quite large - it amounts to more than 4 GB.
To reduce the size of the *n-gram* and make loading faster, `kenLM` allows converting `.arpa` files to binary ones using the `build_binary` executable.

Let's make use of it here.

In [ ]:
!kenlm/build/bin/build_binary MSP-Processor-With-LM-LRS2/language_model/3gram_correct.arpa MSP-Processor-With-LM-LRS2/language_model/3gram.bin

Reading MSP-Processor-With-LM-LRS2/language_model/3gram_correct.arpa
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
SUCCESS


Great, it worked! Let's remove the `.arpa` file and check the size of the binary *5-gram* LM.

In [ ]:
!rm MSP-Processor-With-LM-LRS2/language_model/3gram_correct.arpa && tree -h MSP-Processor-With-LM-LRS2/

[4.0K]  MSP-Processor-With-LM-LRS2/
├── [ 198]  alphabet.json
├── [5.6K]  feature_extraction_msp_audio.py
├── [4.0K]  language_model
│   ├── [ 34M]  3gram.bin
│   ├── [  78]  attrs.json
│   └── [308K]  unigrams.txt
├── [ 26K]  processing_msp_with_lm.py
├── [1.2K]  processor_config.json
├── [1.4K]  tokenizer_config.json
├── [5.3K]  video_processing_msp_visual.py
└── [ 358]  vocab.json

1 directory, 10 files


Nice, we reduced the *n-gram* by more than half to less than 2GB now. In the final step, let's upload all files.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

repo = api.upload_folder(
    repo_id="MahmoodAnaam/MSP-Processor-With-LM-LRS2",
    folder_path="MSP-Processor-With-LM-LRS2",
    commit_message="Upload lm-lrs2 decoder"
)

That's it. Now you should be able to use the *5gram* for LM-boosted decoding as shown in Section 1.

As can be seen on [`xls-r-300m-sv`'s model card](https://huggingface.co/hf-test/xls-r-300m-sv#inference-with-lm) our *5gram* LM-boosted decoder yields a WER of 18.85% on Common Voice's 7 test set which is a relative performance of *ca.* 30% 🔥.

In [ ]:
from transformers import AutoProcessor

processor_with_lm = AutoProcessor.from_pretrained("MahmoodAnaam/MSP-Processor-With-LM-LRS2",trust_remote_code=True)
processor_with_lm

processor_config.json:   0%|          | 0.00/1.23k [00:00<?, ?B/s]

processing_msp_with_lm.py:   0%|          | 0.00/27.0k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/MahmoodAnaam/MSP-Processor-With-LM-LRS2:
- processing_msp_with_lm.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


feature_extraction_msp_audio.py:   0%|          | 0.00/5.74k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/MahmoodAnaam/MSP-Processor-With-LM-LRS2:
- feature_extraction_msp_audio.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

tokenizer_config.json:   0%|          | 0.00/1.45k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/358 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

MSPProcessorWithLM:
- feature_extractor: MSPAudioFeatureExtractor {
  "auto_map": {
    "AutoFeatureExtractor": "feature_extraction_msp_audio.MSPAudioFeatureExtractor",
    "AutoProcessor": "processing_msp_with_lm.MSPProcessorWithLM"
  },
  "do_normalize": true,
  "feature_extractor_type": "MSPAudioFeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- video_processor: MSPVisualVideoProcessor {
  "auto_map": {
    "AutoProcessor": "processing_msp_with_lm.MSPProcessorWithLM",
    "AutoVideoProcessor": "video_processing_msp_visual.MSPVisualVideoProcessor"
  },
  "crop_size": {
    "height": 88,
    "width": 88
  },
  "do_center_crop": true,
  "do_convert_rgb_to_grayscale": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": 0.421,
  "image_std": 0.165,
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "return_metadata": false,
  "size": {
    "he